# Action Pack Roundtrip Dev

Local-only notebook for proving that Action Pack category resolution, bit packing, and bit decode stay in sync with the current `_tallCategories.csv` schema

This notebook mirrors the current app contract:

- Load `answerID`, `category`, and `manualRank` from `_tallCategories.csv`
- Resolve selected answer tokens to ranked unique categories
- Pack those categories into the compact `ap2.` bitmask format
- Decode the packed value back into categories
- Check that roundtrips are stable

In [ ]:
from __future__ import annotations

from collections import defaultdict
from itertools import islice
from pathlib import Path
import csv

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CSV_PATH = REPO_ROOT / 'plugins' / 'emfn-action-pack-plugin' / 'assets' / 'data' / '_tallCategories.csv'
PAYLOAD_PREFIX = 'ap2.'
SEGMENT_SIZE = 31

CSV_PATH

In [ ]:
def load_action_pack_registry(csv_path: Path):
    with csv_path.open(newline='', encoding='utf-8') as handle:
        reader = csv.DictReader(handle)
        rows = list(reader)

    required = {'answerID', 'category', 'manualRank'}
    missing = required - set(reader.fieldnames or [])
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')

    def parse_manual_rank(value):
        text = (value or '').strip()
        if not text:
            return 0
        try:
            return int(text)
        except ValueError:
            return 0

    registry = defaultdict(list)
    category_ranks = {}
    category_positions = {}
    category_order = []

    for row in rows:
        answer_id = (row.get('answerID') or '').strip()
        category_name = (row.get('category') or '').strip()
        manual_rank = parse_manual_rank(row.get('manualRank'))

        if not answer_id or not category_name:
            continue

        existing = next((entry for entry in registry[answer_id] if entry['category'] == category_name), None)
        if existing is None:
            registry[answer_id].append({'category': category_name, 'manual_rank': manual_rank})
        else:
            existing['manual_rank'] = max(existing['manual_rank'], manual_rank)

        if category_name not in category_ranks:
            category_positions[category_name] = len(category_order)
            category_order.append(category_name)
            category_ranks[category_name] = manual_rank
        else:
            category_ranks[category_name] = max(category_ranks[category_name], manual_rank)

    for answer_id, entries in registry.items():
        entries.sort(key=lambda entry: (-entry['manual_rank'], category_positions[entry['category']]))

    category_order.sort(key=lambda category: (-category_ranks[category], category_positions[category]))

    return {
        'rows': rows,
        'registry': dict(registry),
        'category_ranks': category_ranks,
        'category_positions': category_positions,
        'category_order': category_order,
    }

data = load_action_pack_registry(CSV_PATH)
len(data['registry']), len(data['category_order'])

In [ ]:
def resolve_matched_categories(selected_tokens, registry):
    category_ranks = {}
    category_positions = {}
    ordered_categories = []

    for token in selected_tokens:
        for entry in registry.get(token, []):
            category_name = entry['category'].strip()
            manual_rank = int(entry.get('manual_rank', 0))
            if not category_name:
                continue

            if category_name not in category_ranks:
                category_positions[category_name] = len(ordered_categories)
                ordered_categories.append(category_name)
                category_ranks[category_name] = manual_rank
            else:
                category_ranks[category_name] = max(category_ranks[category_name], manual_rank)

    ordered_categories.sort(key=lambda category: (-category_ranks[category], category_positions[category]))
    return ordered_categories

def canonicalize_category_names(category_names, category_order):
    selected = {category for category in category_names if category}
    return [category for category in category_order if category in selected]

def pack_action_pack_bits(category_names, category_order, prefix=PAYLOAD_PREFIX, segment_size=SEGMENT_SIZE):
    index_by_category = {category: index for index, category in enumerate(category_order)}
    packed_segments = []

    for category_name in category_names:
        category_index = index_by_category.get(category_name)
        if category_index is None:
            continue

        segment_index = category_index // segment_size
        bit_index = category_index % segment_size

        while len(packed_segments) <= segment_index:
            packed_segments.append(0)

        packed_segments[segment_index] |= 1 << bit_index

    if not packed_segments:
        packed_segments.append(0)

    return prefix + '.'.join(format(segment, 'x') if False else base36(segment) for segment in packed_segments)

def base36(number: int) -> str:
    alphabet = '0123456789abcdefghijklmnopqrstuvwxyz'
    if number == 0:
        return '0'

    digits = []
    current = number
    while current:
        current, remainder = divmod(current, 36)
        digits.append(alphabet[remainder])
    return ''.join(reversed(digits))

def decode_action_pack_payload(payload, category_order, prefix=PAYLOAD_PREFIX, segment_size=SEGMENT_SIZE):
    if not payload.startswith(prefix):
        return None

    encoded_payload = payload[len(prefix):]
    if not encoded_payload:
        return None

    resolved_categories = []
    for segment_index, segment in enumerate(encoded_payload.split('.')):
        normalized_segment = segment.strip()
        if not normalized_segment or any(ch not in '0123456789abcdefghijklmnopqrstuvwxyz' for ch in normalized_segment):
            return None

        segment_value = int(normalized_segment, 36)
        for bit_index in range(segment_size):
            category_index = segment_index * segment_size + bit_index
            if category_index >= len(category_order):
                continue
            if segment_value & (1 << bit_index):
                resolved_categories.append(category_order[category_index])

    return resolved_categories or None

## Example roundtrip check

This code block runs a small end-to-end test of the Action Pack workflow with three sample answer tokens:

- `mode-before`
- `distribPlatforms-sms`
- `disasterType-flooding`

It then walks through each step of the contract:

1. **Resolve categories**
    - `resolve_matched_categories(sample_tokens, data['registry'])`
    - Looks up the selected tokens in the registry and gathers the matching categories.
    - Categories are deduplicated and ordered by `manualRank`, with original schema position used as a tiebreaker.

2. **Canonicalize category order**
    - `canonicalize_category_names(matched_categories, data['category_order'])`
    - Reorders the resolved categories into the master `category_order`.
    - This produces the expected stable order for packing and decoding.

3. **Pack into the compact payload**
    - `pack_action_pack_bits(matched_categories, data['category_order'])`
    - Converts the selected categories into the `ap2.` bitmask format.
    - In this notebook, the sample payload is `ap2.zik0zj.6bj`.

4. **Decode the payload back to categories**
    - `decode_action_pack_payload(payload, data['category_order'])`
    - Reads the packed bitmask and reconstructs the selected categories.

5. **Verify roundtrip stability**
    - `expected_categories == decoded_categories`
    - Confirms that decoding the packed value returns the same canonical category set.

The final dictionary is just a readable summary of the test inputs, intermediate values, packed payload, decoded output, and whether the roundtrip succeeded.

In [ ]:
import pandas as pd

sample_tokens = ['distribPlatforms-sms', 'disasterType-flooding']
matched_categories = resolve_matched_categories(sample_tokens, data['registry'])
expected_categories = canonicalize_category_names(matched_categories, data['category_order'])
payload = pack_action_pack_bits(matched_categories, data['category_order'])
decoded_categories = decode_action_pack_payload(payload, data['category_order']) or []

max_len = max(len(expected_categories), len(decoded_categories))
comparison = pd.DataFrame({
    'position': range(max_len),
    'before': [expected_categories[i] if i < len(expected_categories) else None for i in range(max_len)],
    'after': [decoded_categories[i] if i < len(decoded_categories) else None for i in range(max_len)],
})
comparison['aligned'] = comparison['before'] == comparison['after']

summary = pd.DataFrame([{
    'sample_tokens': ', '.join(sample_tokens),
    'matched_count': len(matched_categories),
    'expected_count': len(expected_categories),
    'decoded_count': len(decoded_categories),
    'payload': payload,
    'roundtrip_matches': expected_categories == decoded_categories,
}])

display(summary)
comparison

In [ ]:
def run_roundtrip_checks(registry, category_order, category_ranks, max_token_pairs=200):
    token_roundtrip_failures = []
    tokens = list(registry.keys())

    for token in tokens:
        matched = resolve_matched_categories([token], registry)
        expected = canonicalize_category_names(matched, category_order)
        payload = pack_action_pack_bits(matched, category_order)
        decoded = decode_action_pack_payload(payload, category_order) or []
        if expected != decoded:
            token_roundtrip_failures.append({
                'tokens': [token],
                'matched': matched,
                'expected': expected,
                'payload': payload,
                'decoded': decoded,
            })

    pair_failures = []
    tested_pairs = 0
    for index, left in enumerate(tokens):
        for right in tokens[index + 1:]:
            matched = resolve_matched_categories([left, right], registry)
            expected = canonicalize_category_names(matched, category_order)
            payload = pack_action_pack_bits(matched, category_order)
            decoded = decode_action_pack_payload(payload, category_order) or []
            tested_pairs += 1
            if expected != decoded:
                pair_failures.append({
                    'tokens': [left, right],
                    'matched': matched,
                    'expected': expected,
                    'payload': payload,
                    'decoded': decoded,
                })
            if tested_pairs >= max_token_pairs:
                break
        if tested_pairs >= max_token_pairs:
            break

    tie_groups = defaultdict(list)
    for category_name, rank in category_ranks.items():
        tie_groups[rank].append(category_name)

    tied_ranks = {rank: names for rank, names in tie_groups.items() if len(names) > 1}

    return {
        'token_count': len(tokens),
        'category_count': len(category_order),
        'single_token_failures': token_roundtrip_failures,
        'tested_pairs': tested_pairs,
        'pair_failures': pair_failures,
        'tied_rank_groups': tied_ranks,
        'first_20_category_order': list(islice(category_order, 20)),
    }


def failures_to_frame(failures):
    if not failures:
        return pd.DataFrame(columns=['tokens', 'matched', 'expected', 'payload', 'decoded'])

    return pd.DataFrame([{
        'tokens': ', '.join(item['tokens']),
        'matched': ', '.join(item['matched']),
        'expected': ', '.join(item['expected']),
        'payload': item['payload'],
        'decoded': ', '.join(item['decoded']),
    } for item in failures])


def style_wrapped_table(df, wrap_columns):
    wrap_columns = [column for column in wrap_columns if column in df.columns]
    styler = df.style

    if wrap_columns:
        styler = styler.set_properties(
            subset=wrap_columns,
            **{
                'white-space': 'normal',
                'word-break': 'break-word',
                'overflow-wrap': 'anywhere',
                'max-width': 'none',
            }
        )

    return styler.set_properties(
        **{
            'vertical-align': 'top',
        }
    )


report = run_roundtrip_checks(
    data['registry'],
    data['category_order'],
    data['category_ranks'],
)

report_summary = pd.DataFrame([{
    'token_count': report['token_count'],
    'category_count': report['category_count'],
    'single_token_failures': len(report['single_token_failures']),
    'tested_pairs': report['tested_pairs'],
    'pair_failures': len(report['pair_failures']),
    'tied_rank_group_count': len(report['tied_rank_groups']),
    'first_20_category_order': ', '.join(report['first_20_category_order']),
}])

tied_rank_groups_df = pd.DataFrame([
    {
        'manual_rank': rank,
        'category_count': len(names),
        'categories': ', '.join(names),
    }
    for rank, names in sorted(report['tied_rank_groups'].items(), key=lambda item: item[0], reverse=True)
])

single_token_failures_table = failures_to_frame(report['single_token_failures'])
pair_failures_table = failures_to_frame(report['pair_failures'])

report_summary_table = style_wrapped_table(
    report_summary,
    ['first_20_category_order'],
)

tied_rank_groups_table = style_wrapped_table(
    tied_rank_groups_df,
    ['categories'],
)

single_token_failures_styled = style_wrapped_table(
    single_token_failures_table,
    ['tokens', 'matched', 'expected', 'decoded'],
)

pair_failures_styled = style_wrapped_table(
    pair_failures_table,
    ['tokens', 'matched', 'expected', 'decoded'],
)

print('Report summary: high-level counts for tokens, categories, failures, and tied rank groups')
display(report_summary_table)

print('Tied rank groups: categories that currently share the same manual rank')
display(tied_rank_groups_table)

print('Single-token roundtrip failures: any individual tokens whose packed payload does not decode to the expected categories')
display(single_token_failures_styled)

print('Pair roundtrip failures: any token pairs whose packed payload does not decode to the expected categories')
display(pair_failures_styled)